# LeetCode 1600 — Throne Inheritance bằng Python

Link đề: https://leetcode.com/problems/throne-inheritance/ · Độ khó: **Medium** · Chủ đề: Hash Table, Tree, Depth-First Search, Design

Skeleton LeetCode:

```python
class ThroneInheritance:

    def __init__(self, kingName: str):
        ...

    def birth(self, parentName: str, childName: str) -> None:
        ...

    def death(self, name: str) -> None:
        ...

    def getInheritanceOrder(self) -> List[str]:
        ...
```

## 1. Mục tiêu học

1. Đọc hiểu một hàm đệ quy cho trong đề (`Successor`) và nhận ra nó **chính là duyệt tiền thứ tự (preorder DFS)** của cây gia phả.
2. Thiết kế lớp với các cấu trúc dữ liệu phù hợp: `dict` tên → danh sách con **theo thứ tự sinh**, `set` người đã chết.
3. Hiểu nguyên tắc "**đánh dấu chết, không xóa khỏi cây**" — người chết vẫn là "cầu nối" tới con cháu.
4. Viết DFS bằng **đệ quy** và bằng **ngăn xếp tường minh** (tránh vượt giới hạn đệ quy khi cây là một chuỗi dài 10⁵ đời).
5. Dựng **bộ chạy chuỗi lệnh giả lập** kiểu LeetCode để kiểm thử lớp thiết kế.

## 2. Đề bài & ví dụ

Một vương quốc gồm nhà vua, các con, các cháu, … Thỉnh thoảng có người chết hoặc có trẻ được sinh ra.

**Thứ tự kế vị** bắt đầu bằng nhà vua. Định nghĩa hàm đệ quy `Successor(x, curOrder)` trả về người kế tiếp sau `x` khi thứ tự hiện có là `curOrder`:

```
Successor(x, curOrder):
    nếu x không có con hoặc mọi con của x đã có trong curOrder:
        nếu x là vua: trả null
        ngược lại: trả Successor(cha của x, curOrder)
    ngược lại: trả người con lớn tuổi nhất của x chưa có trong curOrder
```

Ví dụ: vua có hai con Alice và Bob (Alice lớn hơn), Alice có con Jack. Bắt đầu `curOrder = ["king"]` → `Successor(king) = Alice` → `Successor(Alice) = Jack` → `Successor(Jack) = Bob` → `Successor(Bob) = null`. Thứ tự: `["king", "Alice", "Jack", "Bob"]`.

Cài đặt lớp `ThroneInheritance`:
- `ThroneInheritance(kingName)`: khởi tạo với tên vua.
- `birth(parentName, childName)`: `parentName` sinh ra `childName`.
- `death(name)`: `name` chết. Việc chết **không ảnh hưởng** hàm `Successor` hay thứ tự kế vị — chỉ **đánh dấu** là đã chết.
- `getInheritanceOrder()`: trả danh sách thứ tự kế vị hiện tại, **bỏ qua người đã chết**.

**Ví dụ 1:**
```
["ThroneInheritance","birth","birth","birth","birth","birth","birth","getInheritanceOrder","death","getInheritanceOrder"]
[["king"],["king","andy"],["king","bob"],["king","catherine"],["andy","matthew"],["bob","alex"],["bob","asha"],[null],["bob"],[null]]
→ [null,null,null,null,null,null,null,["king","andy","matthew","bob","alex","asha","catherine"],null,["king","andy","matthew","alex","asha","catherine"]]
```

**Ràng buộc:**
- Độ dài các tên từ 1 đến 15, chỉ chữ thường.
- Mọi `childName` và `kingName` đôi một khác nhau.
- Tên trong `death` luôn đã xuất hiện trước (là vua hoặc đã được sinh).
- Mỗi lần `birth(parentName, childName)`, `parentName` đảm bảo còn sống.
- Tối đa `10⁵` lần gọi `birth` và `death`; tối đa `10` lần gọi `getInheritanceOrder`.

In [1]:
import random
import sys
import time
from collections import defaultdict
from typing import List

EX1_OPS = ["ThroneInheritance", "birth", "birth", "birth", "birth", "birth", "birth",
           "getInheritanceOrder", "death", "getInheritanceOrder"]
EX1_ARGS = [["king"], ["king", "andy"], ["king", "bob"], ["king", "catherine"], ["andy", "matthew"],
            ["bob", "alex"], ["bob", "asha"], [], ["bob"], []]
EX1_OUT = [None, None, None, None, None, None, None,
           ["king", "andy", "matthew", "bob", "alex", "asha", "catherine"], None,
           ["king", "andy", "matthew", "alex", "asha", "catherine"]]

def run_commands(cls, ops, args):
    """Bộ chạy lệnh GIẢ LẬP kiểu LeetCode: lệnh đầu là constructor, các lệnh sau gọi method."""
    obj, out = None, []
    for op, a in zip(ops, args):
        if op == cls.__name__ or obj is None:
            obj = cls(*a); out.append(None)
        else:
            out.append(getattr(obj, op)(*a))
    return out

print("Bộ chạy lệnh giả lập đã sẵn sàng; số lệnh ví dụ 1:", len(EX1_OPS))

Bộ chạy lệnh giả lập đã sẵn sàng; số lệnh ví dụ 1: 10


## 3. Kiến thức nền

### 3.1 `defaultdict(list)` — từ điển tên → danh sách con

`defaultdict(list)` tự tạo `[]` khi truy cập khóa mới. `append` giữ đúng **thứ tự sinh** (con lớn tuổi đứng trước).

In [2]:
children = defaultdict(list)
children["king"].append("andy")
children["king"].append("bob")
children["andy"].append("matthew")
print(dict(children))
print("con của bob:", children["bob"], "(tự tạo list rỗng)")

{'king': ['andy', 'bob'], 'andy': ['matthew']}
con của bob: [] (tự tạo list rỗng)


### 3.2 `set` cho tập người đã chết

Kiểm tra `name in dead` trung bình O(1). Không cần xóa ai khỏi cây.

In [3]:
dead = set()
dead.add("bob")
print("bob" in dead, "andy" in dead)

True False


### 3.3 Duyệt tiền thứ tự (preorder): gốc trước, rồi lần lượt từng cây con từ trái sang phải

In [4]:
tree = {"A": ["B", "C"], "B": ["D"], "C": [], "D": []}
def preorder(x, out):
    out.append(x)
    for c in tree[x]:
        preorder(c, out)
    return out
print(preorder("A", []))

['A', 'B', 'D', 'C']


### 3.4 Giới hạn đệ quy của Python

Mặc định khoảng 1000 tầng. Một cây "chuỗi" (mỗi người sinh đúng một con, `10⁵` đời) sẽ làm DFS đệ quy vượt giới hạn ⇒ `RecursionError`. Cách an toàn: dùng **ngăn xếp tường minh**. Khi đẩy các con vào stack, phải đẩy theo **thứ tự ngược** để con lớn nhất được lấy ra trước.

In [5]:
print("recursion limit:", sys.getrecursionlimit())
stack, out = ["A"], []
while stack:
    x = stack.pop()
    out.append(x)
    stack.extend(reversed(tree[x]))      # đẩy ngược để con đầu tiên ra trước
print("preorder bằng stack:", out)

recursion limit: 1000
preorder bằng stack: ['A', 'B', 'D', 'C']


## 4. Ý tưởng nền tảng / trực giác

Chạy tay hàm `Successor` trên cây của ví dụ 1 (trước khi bob chết):

```
king
├── andy
│   └── matthew
├── bob
│   ├── alex
│   └── asha
└── catherine
```

- `Successor(king)`: con lớn nhất chưa có là `andy`.
- `Successor(andy)`: con chưa có là `matthew`.
- `Successor(matthew)`: không con ⇒ lên cha `andy` ⇒ andy hết con chưa có ⇒ lên `king` ⇒ con chưa có lớn nhất là `bob`.
- … tiếp tục: `alex`, `asha`, `catherine`, rồi `null`.

Thứ tự `king, andy, matthew, bob, alex, asha, catherine` chính là **preorder**: "thăm mình, rồi thăm hết nhánh con cả, rồi nhánh con thứ…".

Khi `bob` chết: cây **không đổi**, chỉ là khi xuất danh sách ta bỏ tên `bob`. Con của bob (`alex`, `asha`) vẫn đứng đúng chỗ.

## 5. Cách 1 — Brute Force / baseline: cài đặt đúng nguyên văn `Successor`

**Ý tưởng:** lưu cha và danh sách con. `getInheritanceOrder` bắt đầu từ `curOrder = [king]`, gọi `Successor(người cuối, curOrder)` cho đến khi trả `None`; cuối cùng lọc người chết. Để "chưa có trong curOrder" kiểm tra nhanh, dùng thêm một `set`.

In [6]:
class ThroneBrute:
    def __init__(self, kingName):
        self.king = kingName
        self.parent = {kingName: None}
        self.children = defaultdict(list)
        self.dead = set()

    def birth(self, parentName, childName):
        self.parent[childName] = parentName
        self.children[parentName].append(childName)

    def death(self, name):
        self.dead.add(name)

    def _successor(self, x, in_order):
        while True:                                   # khử đệ quy "lên cha"
            for c in self.children[x]:
                if c not in in_order:
                    return c                          # con lớn nhất chưa có
            if x == self.king:
                return None
            x = self.parent[x]

    def getInheritanceOrder(self):
        order, in_order = [self.king], {self.king}
        while True:
            nxt = self._successor(order[-1], in_order)
            if nxt is None:
                break
            order.append(nxt); in_order.add(nxt)
        return [p for p in order if p not in self.dead]

res = run_commands(ThroneBrute, EX1_OPS, EX1_ARGS)
print(res[7]); print(res[9]); print("khớp:", res == EX1_OUT)

['king', 'andy', 'matthew', 'bob', 'alex', 'asha', 'catherine']
['king', 'andy', 'matthew', 'alex', 'asha', 'catherine']
khớp: True


**Độ phức tạp:** mỗi lần gọi `_successor` có thể leo lên nhiều tầng và quét lại danh sách con từ đầu. Worst case (vua có `n` con: mỗi lần quét lại các con đã có) ⇒ O(n²) cho một lần `getInheritanceOrder`. `birth`, `death` O(1).

**Nút thắt:** việc "quét con chưa có trong curOrder" lặp đi lặp lại. Đo thử với vua có nhiều con:

In [7]:
for n in [1000, 2000, 4000]:
    t = ThroneBrute("k")
    for i in range(n):
        t.birth("k", f"c{i}")
    t0 = time.perf_counter(); t.getInheritanceOrder()
    print(f"vua có {n} con: {time.perf_counter() - t0:.3f}s")

vua có 1000 con: 0.010s
vua có 2000 con: 0.037s


vua có 4000 con: 0.148s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** `Successor` mô tả đúng quy trình DFS tiền thứ tự: "đi xuống con lớn nhất chưa thăm; hết con thì quay lên cha". Vì vậy thứ tự kế vị = **preorder của cây** với các con xếp theo thứ tự sinh. Không cần mô phỏng `Successor` — chỉ cần DFS.

**Quy tắc:**
- `birth(p, c)`: `children[p].append(c)` — O(1), tự động giữ thứ tự sinh.
- `death(x)`: `dead.add(x)` — O(1), **không** động vào cây.
- `getInheritanceOrder()`: DFS preorder từ vua; **nếu** người đang thăm chưa chết **thì** thêm vào kết quả; **luôn** đi tiếp vào các con (kể cả khi người đó đã chết).

**Invariant:** `children[x]` luôn là danh sách con của `x` theo thứ tự sinh; cây chỉ lớn thêm, không bao giờ mất nút.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Cây `dict` + DFS đệ quy

Ngắn gọn nhất. Nhược điểm: cây sâu (chuỗi dài) có thể vượt giới hạn đệ quy (xem Lỗi 3); trên LeetCode có thể phải tăng `sys.setrecursionlimit`.

In [8]:
class ThroneRecursive:
    def __init__(self, kingName):
        self.king = kingName
        self.children = defaultdict(list)
        self.dead = set()

    def birth(self, parentName, childName):
        self.children[parentName].append(childName)

    def death(self, name):
        self.dead.add(name)

    def getInheritanceOrder(self):
        order = []
        def dfs(x):
            if x not in self.dead:
                order.append(x)
            for c in self.children[x]:          # vẫn đi qua người đã chết
                dfs(c)
        dfs(self.king)
        return order

res = run_commands(ThroneRecursive, EX1_OPS, EX1_ARGS)
print(res[9], "| khớp:", res == EX1_OUT)

['king', 'andy', 'matthew', 'alex', 'asha', 'catherine'] | khớp: True


Code nộp LeetCode của Cách 2:

```python
import sys
from collections import defaultdict

class ThroneInheritance:
    def __init__(self, kingName: str):
        sys.setrecursionlimit(300000)
        self.king = kingName
        self.children = defaultdict(list)
        self.dead = set()

    def birth(self, parentName: str, childName: str) -> None:
        self.children[parentName].append(childName)

    def death(self, name: str) -> None:
        self.dead.add(name)

    def getInheritanceOrder(self) -> List[str]:
        order = []
        def dfs(x):
            if x not in self.dead:
                order.append(x)
            for c in self.children[x]:
                dfs(c)
        dfs(self.king)
        return order
```

**Độ phức tạp:** `birth`/`death` O(1); `getInheritanceOrder` O(N) với `N` = tổng số người; Space O(N) cho cây + O(độ sâu) ngăn xếp đệ quy.

### 7.2 Cách 3 — Cây `dict` + DFS bằng ngăn xếp tường minh

Thay đệ quy bằng stack: lấy `x` ra, ghi nếu còn sống, rồi đẩy các con **theo thứ tự ngược** (`reversed`) để con lớn nhất nằm trên đỉnh. Không có giới hạn độ sâu.

In [9]:
class ThroneStack:
    def __init__(self, kingName):
        self.king = kingName
        self.children = defaultdict(list)
        self.dead = set()

    def birth(self, parentName, childName):
        self.children[parentName].append(childName)

    def death(self, name):
        self.dead.add(name)

    def getInheritanceOrder(self):
        order, stack = [], [self.king]
        while stack:
            x = stack.pop()
            if x not in self.dead:
                order.append(x)
            stack.extend(reversed(self.children[x]))
        return order

res = run_commands(ThroneStack, EX1_OPS, EX1_ARGS)
print(res[9], "| khớp:", res == EX1_OUT)

['king', 'andy', 'matthew', 'alex', 'asha', 'catherine'] | khớp: True


Code nộp LeetCode của Cách 3: xem mục 9 (cách được chọn để nộp).

**Độ phức tạp:** như Cách 2 nhưng không phụ thuộc giới hạn đệ quy: `getInheritanceOrder` O(N) thời gian, O(N) bộ nhớ (stack tối đa O(N)).

### 7.3 Cách 4 — Đi theo đúng định nghĩa `Successor` nhưng hiệu quả: con trỏ cha + chỉ số con kế tiếp

Đây là cách "trung thành với đề" mà vẫn O(N): thay vì kiểm tra "con nào chưa có trong `curOrder`", mỗi nút nhớ **chỉ số con kế tiếp chưa thăm** `nxt[x]`. Bắt đầu từ vua:
- nếu `x` còn con chưa thăm (`nxt[x] < len(children[x])`) ⇒ đi xuống con đó, tăng `nxt[x]`;
- ngược lại ⇒ lên cha (`parent[x]`); nếu `x` là vua thì dừng.

Mỗi cạnh được đi xuống một lần và đi lên một lần ⇒ O(N), bộ nhớ phụ O(N) cho `nxt` nhưng **không cần stack**.

In [10]:
class ThroneParentWalk:
    def __init__(self, kingName):
        self.king = kingName
        self.parent = {kingName: None}
        self.children = defaultdict(list)
        self.dead = set()

    def birth(self, parentName, childName):
        self.parent[childName] = parentName
        self.children[parentName].append(childName)

    def death(self, name):
        self.dead.add(name)

    def getInheritanceOrder(self):
        nxt = defaultdict(int)
        order = []
        x = self.king
        if x not in self.dead:
            order.append(x)
        while x is not None:
            kids = self.children[x]
            if nxt[x] < len(kids):               # còn con chưa thăm: đi xuống
                c = kids[nxt[x]]; nxt[x] += 1
                if c not in self.dead:
                    order.append(c)
                x = c
            else:                                # hết con: lên cha (vua có cha None -> dừng)
                x = self.parent[x]
        return order

res = run_commands(ThroneParentWalk, EX1_OPS, EX1_ARGS)
print(res[9], "| khớp:", res == EX1_OUT)

['king', 'andy', 'matthew', 'alex', 'asha', 'catherine'] | khớp: True


Code nộp LeetCode của Cách 4:

```python
from collections import defaultdict

class ThroneInheritance:
    def __init__(self, kingName: str):
        self.king = kingName
        self.parent = {kingName: None}
        self.children = defaultdict(list)
        self.dead = set()

    def birth(self, parentName: str, childName: str) -> None:
        self.parent[childName] = parentName
        self.children[parentName].append(childName)

    def death(self, name: str) -> None:
        self.dead.add(name)

    def getInheritanceOrder(self) -> List[str]:
        nxt = defaultdict(int)
        order = [] if self.king in self.dead else [self.king]
        x = self.king
        while x is not None:
            kids = self.children[x]
            if nxt[x] < len(kids):
                c = kids[nxt[x]]; nxt[x] += 1
                if c not in self.dead:
                    order.append(c)
                x = c
            else:
                x = self.parent[x]
        return order
```

**Độ phức tạp:** `getInheritanceOrder` O(N); `birth`/`death` O(1); Space O(N).

## 8. TRACE từng bước

Trace Cách 3 (stack) trên ví dụ 1 sau khi `bob` chết:

| bước | pop | còn sống? | order | đẩy (ngược) | stack sau |
|---|---|---|---|---|---|
| 1 | king | có | [king] | catherine, bob, andy | [catherine, bob, andy] |
| 2 | andy | có | +andy | matthew | [catherine, bob, matthew] |
| 3 | matthew | có | +matthew | — | [catherine, bob] |
| 4 | bob | **chết** | (bỏ qua) | asha, alex | [catherine, asha, alex] |
| 5 | alex | có | +alex | — | [catherine, asha] |
| 6 | asha | có | +asha | — | [catherine] |
| 7 | catherine | có | +catherine | — | [] |

In [11]:
def trace_stack_order(throne):
    order, stack = [], [throne.king]
    step = 0
    while stack:
        x = stack.pop(); step += 1
        alive = x not in throne.dead
        if alive:
            order.append(x)
        kids = list(reversed(throne.children[x]))
        stack.extend(kids)
        print(f"bước {step}: pop {x:9s} {'sống' if alive else 'CHẾT'}  đẩy {kids}  stack={stack}  order={order}")
    return order

t = ThroneStack("king")
for p, c in [("king", "andy"), ("king", "bob"), ("king", "catherine"), ("andy", "matthew"), ("bob", "alex"), ("bob", "asha")]:
    t.birth(p, c)
t.death("bob")
trace_stack_order(t)

bước 1: pop king      sống  đẩy ['catherine', 'bob', 'andy']  stack=['catherine', 'bob', 'andy']  order=['king']
bước 2: pop andy      sống  đẩy ['matthew']  stack=['catherine', 'bob', 'matthew']  order=['king', 'andy']
bước 3: pop matthew   sống  đẩy []  stack=['catherine', 'bob']  order=['king', 'andy', 'matthew']
bước 4: pop bob       CHẾT  đẩy ['asha', 'alex']  stack=['catherine', 'asha', 'alex']  order=['king', 'andy', 'matthew']
bước 5: pop alex      sống  đẩy []  stack=['catherine', 'asha']  order=['king', 'andy', 'matthew', 'alex']
bước 6: pop asha      sống  đẩy []  stack=['catherine']  order=['king', 'andy', 'matthew', 'alex', 'asha']
bước 7: pop catherine sống  đẩy []  stack=[]  order=['king', 'andy', 'matthew', 'alex', 'asha', 'catherine']


['king', 'andy', 'matthew', 'alex', 'asha', 'catherine']

In [12]:
# Trace Cách 4: đi xuống / đi lên theo định nghĩa Successor
def trace_parent_walk(throne):
    nxt = defaultdict(int); x = throne.king
    order = [] if x in throne.dead else [x]
    print(f"bắt đầu tại {x}")
    while x is not None:
        kids = throne.children[x]
        if nxt[x] < len(kids):
            c = kids[nxt[x]]; nxt[x] += 1
            if c not in throne.dead: order.append(c)
            print(f"  {x} -> xuống con {c}{' (chết, không ghi)' if c in throne.dead else ''}")
            x = c
        else:
            print(f"  {x} hết con -> lên {throne.parent[x]}")
            x = throne.parent[x]
    print("order =", order)
    return order

t4 = ThroneParentWalk("king")
for p, c in [("king", "alice"), ("king", "bob"), ("alice", "jack")]:
    t4.birth(p, c)
trace_parent_walk(t4)

bắt đầu tại king
  king -> xuống con alice
  alice -> xuống con jack
  jack hết con -> lên alice
  alice hết con -> lên king
  king -> xuống con bob
  bob hết con -> lên king
  king hết con -> lên None
order = ['king', 'alice', 'jack', 'bob']


['king', 'alice', 'jack', 'bob']

## 9. Code nộp LeetCode

```python
from collections import defaultdict
from typing import List

class ThroneInheritance:

    def __init__(self, kingName: str):
        self.king = kingName
        self.children = defaultdict(list)   # tên -> các con theo thứ tự sinh
        self.dead = set()

    def birth(self, parentName: str, childName: str) -> None:
        self.children[parentName].append(childName)

    def death(self, name: str) -> None:
        self.dead.add(name)                  # chỉ đánh dấu, không xóa khỏi cây

    def getInheritanceOrder(self) -> List[str]:
        order, stack = [], [self.king]
        while stack:
            x = stack.pop()
            if x not in self.dead:
                order.append(x)
            stack.extend(reversed(self.children[x]))
        return order
```

In [13]:
class ThroneInheritance:

    def __init__(self, kingName: str):
        self.king = kingName
        self.children = defaultdict(list)
        self.dead = set()

    def birth(self, parentName: str, childName: str) -> None:
        self.children[parentName].append(childName)

    def death(self, name: str) -> None:
        self.dead.add(name)

    def getInheritanceOrder(self) -> List[str]:
        order, stack = [], [self.king]
        while stack:
            x = stack.pop()
            if x not in self.dead:
                order.append(x)
            stack.extend(reversed(self.children[x]))
        return order

res = run_commands(ThroneInheritance, EX1_OPS, EX1_ARGS)
for op, r in zip(EX1_OPS, res):
    print(f"{op:20s} -> {r}")
print("khớp đầu ra mẫu:", res == EX1_OUT)

ThroneInheritance    -> None
birth                -> None
birth                -> None
birth                -> None
birth                -> None
birth                -> None
birth                -> None
getInheritanceOrder  -> ['king', 'andy', 'matthew', 'bob', 'alex', 'asha', 'catherine']
death                -> None
getInheritanceOrder  -> ['king', 'andy', 'matthew', 'alex', 'asha', 'catherine']
khớp đầu ra mẫu: True


## 10. Vì sao thuật toán đúng

1. **`Successor` = preorder.** Quy nạp: giả sử thứ tự hiện tại là tiền tố của preorder, và người cuối là `x`. Trong preorder, người đứng sau `x` là con lớn nhất của `x` (nếu có); nếu `x` là lá (hoặc các con đã thăm hết) thì là người đứng sau **toàn bộ cây con** của `x`, tức "con kế tiếp chưa thăm" của tổ tiên gần nhất còn con chưa thăm. Đó đúng là điều `Successor` làm: lên cha cho đến khi tìm được con chưa có trong `curOrder`.
2. **Con theo thứ tự sinh:** `append` khi `birth` ⇒ `children[x]` xếp từ lớn đến nhỏ tuổi; DFS duyệt theo thứ tự đó (Cách 3 dùng `reversed` để bù tính LIFO của stack).
3. **Người chết:** đề nói cái chết không ảnh hưởng `Successor` ⇒ cây (và preorder) giữ nguyên; ta chỉ lọc tên chết khi xuất kết quả. Vì vẫn đi vào con của người chết, con cháu họ giữ đúng vị trí.
4. **Mọi người được thăm đúng một lần:** cây không có chu trình, mỗi nút chỉ được đẩy vào stack bởi cha của nó.

## 11. Phân tích Time / Space

Gọi `N` = tổng số người (≤ 10⁵ + 1), `Q ≤ 10` = số lần gọi `getInheritanceOrder`.

| Cách | birth | death | getInheritanceOrder | Space |
|---|---|---|---|---|
| 1. Successor nguyên văn | O(1) | O(1) | O(N²) worst | O(N) |
| 2. DFS đệ quy | O(1) | O(1) | O(N) (cần độ sâu đệ quy O(N)) | O(N) |
| 3. DFS stack | O(1) | O(1) | O(N) | O(N) |
| 4. Con trỏ cha | O(1) | O(1) | O(N) | O(N) |

Tổng cộng: O(số lệnh + Q·N) ≈ `10⁵ + 10·10⁵` — rất nhẹ. Đo với `10⁵` người:

In [14]:
random.seed(0)
names = ["k"] + [f"p{i}" for i in range(1, 100001)]
plan = [(names[random.randint(0, i - 1)], names[i]) for i in range(1, len(names))]
for cls in [ThroneStack, ThroneParentWalk]:
    t0 = time.perf_counter()
    obj = cls("k")
    for p, c in plan:
        obj.birth(p, c)
    for x in random.sample(names, 30000):
        obj.death(x)
    order = obj.getInheritanceOrder()
    print(f"{cls.__name__:17s} {len(order)} người còn sống, {time.perf_counter() - t0:.3f}s")

ThroneStack       70001 người còn sống, 0.104s


ThroneParentWalk  70001 người còn sống, 0.123s


## 12. Bộ test

Dùng bộ chạy lệnh giả lập: ví dụ đề, vua chết, chuỗi sâu `10⁵` đời (Cách 3, 4 không đệ quy), sinh con sau khi đã có người chết, và chuỗi lệnh ngẫu nhiên đối chiếu 4 cách (Cách 1 làm chuẩn). Ràng buộc "cha còn sống khi sinh" được tôn trọng trong bộ sinh ngẫu nhiên.

In [15]:
CLASSES = [ThroneBrute, ThroneRecursive, ThroneStack, ThroneParentWalk, ThroneInheritance]
for cls in CLASSES:
    assert run_commands(cls, EX1_OPS, EX1_ARGS) == EX1_OUT, cls
print("Ví dụ 1: OK cho", len(CLASSES), "cách")

# Vua chết: vẫn phải liệt kê con cháu
ops = ["ThroneInheritance", "birth", "birth", "death", "getInheritanceOrder", "birth", "getInheritanceOrder"]
args = [["king"], ["king", "a"], ["a", "b"], ["king"], [], ["king", "c"], []]
want = [None, None, None, None, ["a", "b"], None, ["a", "b", "c"]]
for cls in CLASSES:
    assert run_commands(cls, ops, args) == want, cls
print("Vua chết: OK")

# Chỉ có vua
for cls in CLASSES:
    assert run_commands(cls, ["ThroneInheritance", "getInheritanceOrder"], [["solo"], []]) == [None, ["solo"]]
print("Chỉ có vua: OK")

Ví dụ 1: OK cho 5 cách
Vua chết: OK
Chỉ có vua: OK


In [16]:
def random_script(rnd, n_ops):
    ops, args = ["ThroneInheritance"], [["king"]]
    alive, everyone, cnt = ["king"], ["king"], 0
    alive_set = {"king"}
    for _ in range(n_ops):
        x = rnd.random()
        if x < 0.55 and alive:
            p = rnd.choice(alive); cnt += 1; c = f"n{cnt}"
            ops.append("birth"); args.append([p, c]); alive.append(c); alive_set.add(c); everyone.append(c)
        elif x < 0.85:
            v = rnd.choice(everyone)
            ops.append("death"); args.append([v])
            if v in alive_set:
                alive_set.discard(v); alive.remove(v)
        else:
            ops.append("getInheritanceOrder"); args.append([])
    ops.append("getInheritanceOrder"); args.append([])
    return ops, args

rnd = random.Random(2024)
for _ in range(400):
    ops, args = random_script(rnd, rnd.randint(1, 40))
    want = run_commands(ThroneBrute, ops, args)
    for cls in CLASSES[1:]:
        assert run_commands(cls, ops, args) == want, (cls, ops, args)
print("400 kịch bản ngẫu nhiên: các cách khớp nhau")

# Chuỗi sâu 1e5 đời: chỉ các cách không đệ quy
for cls in [ThroneStack, ThroneParentWalk, ThroneInheritance]:
    obj = cls("g0")
    for i in range(1, 100001):
        obj.birth(f"g{i-1}", f"g{i}")
    obj.death("g5")
    od = obj.getInheritanceOrder()
    assert len(od) == 100000 and od[:6] == ["g0", "g1", "g2", "g3", "g4", "g6"] and od[-1] == "g100000"
print("Chuỗi 1e5 đời: OK (stack / con trỏ cha)")

400 kịch bản ngẫu nhiên: các cách khớp nhau


Chuỗi 1e5 đời: OK (stack / con trỏ cha)


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Xóa người chết khỏi cây

In [17]:
class WrongDelete(ThroneStack):
    def death(self, name):
        # SAI: gỡ người chết khỏi danh sách con của cha -> mất luôn cả nhánh con cháu
        for p in list(self.children):
            if name in self.children[p]:
                self.children[p].remove(name)

res = run_commands(WrongDelete, EX1_OPS, EX1_ARGS)
print("sai :", res[9])
print("đúng:", EX1_OUT[9])

sai : ['king', 'andy', 'matthew', 'catherine']
đúng: ['king', 'andy', 'matthew', 'alex', 'asha', 'catherine']


**Vì sao sai:** khi `bob` chết, các con `alex`, `asha` vẫn có quyền kế vị (đứng sau `matthew`). Gỡ `bob` khỏi cây làm "đứt dây", mất cả nhánh. **Sửa:** chỉ `dead.add(name)`.

### Lỗi 2 — DFS dừng ở người chết (không đi vào con của họ)

In [18]:
class WrongSkipSubtree(ThroneStack):
    def getInheritanceOrder(self):
        order, stack = [], [self.king]
        while stack:
            x = stack.pop()
            if x in self.dead:
                continue                       # SAI: bỏ luôn cả con cháu
            order.append(x)
            stack.extend(reversed(self.children[x]))
        return order

res = run_commands(WrongSkipSubtree, EX1_OPS, EX1_ARGS)
print("sai :", res[9])
print("đúng:", EX1_OUT[9])

sai : ['king', 'andy', 'matthew', 'catherine']
đúng: ['king', 'andy', 'matthew', 'alex', 'asha', 'catherine']


**Vì sao sai:** `continue` trước khi đẩy con vào stack ⇒ nhánh của `bob` biến mất. **Sửa:** quyết định "có ghi tên không" tách biệt với "có đi vào con không" — **luôn** đi vào con.

### Lỗi 3 — DFS đệ quy trên cây rất sâu

In [19]:
obj = ThroneRecursive("g0")
for i in range(1, 5001):
    obj.birth(f"g{i-1}", f"g{i}")
try:
    obj.getInheritanceOrder()
    print("không lỗi")
except RecursionError as e:
    print("ThroneRecursive trên chuỗi 5000 đời -> RecursionError:", str(e)[:40])

obj2 = ThroneStack("g0")
for i in range(1, 5001):
    obj2.birth(f"g{i-1}", f"g{i}")
print("ThroneStack:", len(obj2.getInheritanceOrder()), "người")

ThroneRecursive trên chuỗi 5000 đời -> RecursionError: maximum recursion depth exceeded
ThroneStack: 5001 người


**Vì sao sai:** mỗi đời thêm một tầng đệ quy; giới hạn mặc định ~1000 ⇒ vỡ ngăn xếp. Ràng buộc cho phép `10⁵` lần `birth` nên một chuỗi `10⁵` đời là hợp lệ. **Sửa:** dùng stack tường minh (Cách 3) hoặc con trỏ cha (Cách 4); nếu vẫn muốn đệ quy thì `sys.setrecursionlimit` (rủi ro tràn stack C).

### Lỗi 4 — Sắp xếp con theo tên thay vì theo thứ tự sinh

In [20]:
class WrongSorted(ThroneStack):
    def birth(self, parentName, childName):
        self.children[parentName].append(childName)
        self.children[parentName].sort()       # SAI: thứ tự kế vị theo tuổi, không theo bảng chữ cái

ops = ["ThroneInheritance", "birth", "birth", "getInheritanceOrder"]
args = [["king"], ["king", "zoe"], ["king", "adam"], []]
print("sai :", run_commands(WrongSorted, ops, args)[-1])
print("đúng:", run_commands(ThroneInheritance, ops, args)[-1])

sai : ['king', 'adam', 'zoe']
đúng: ['king', 'zoe', 'adam']


**Vì sao sai:** `zoe` sinh trước nên lớn tuổi hơn `adam` ⇒ đứng trước. Sắp xếp theo tên phá thứ tự này. (Tương tự, dùng `set` cho danh sách con cũng sai vì `set` không giữ thứ tự chèn.) **Sửa:** chỉ `append`.

### Lỗi 5 — Quên `reversed` khi đẩy con vào stack

In [21]:
class WrongNoReverse(ThroneStack):
    def getInheritanceOrder(self):
        order, stack = [], [self.king]
        while stack:
            x = stack.pop()
            if x not in self.dead:
                order.append(x)
            stack.extend(self.children[x])     # SAI: con út sẽ được pop trước
        return order

res = run_commands(WrongNoReverse, EX1_OPS, EX1_ARGS)
print("sai :", res[7])
print("đúng:", EX1_OUT[7])

sai : ['king', 'catherine', 'bob', 'asha', 'alex', 'andy', 'matthew']
đúng: ['king', 'andy', 'matthew', 'bob', 'alex', 'asha', 'catherine']


**Vì sao sai:** stack là LIFO ⇒ phần tử đẩy sau cùng (con út) ra trước ⇒ thứ tự bị đảo ở mọi tầng. **Sửa:** `stack.extend(reversed(children[x]))`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề cho một **hàm đệ quy "đi xuống con chưa thăm, hết thì lên cha"** ⇒ đó là DFS (preorder). Đừng mô phỏng nguyên văn.
- **Xóa "mềm" (lazy deletion):** khi phần tử bị xóa vẫn còn vai trò cấu trúc (ở đây: nối tới con cháu), chỉ đánh dấu bằng `set`.
- Bài thiết kế: tính tần suất gọi từng hàm (ở đây `birth/death` tới `10⁵`, `getInheritanceOrder` ≤ 10) ⇒ làm `birth/death` O(1), chấp nhận `getInheritanceOrder` O(N).
- Cây có thể sâu `10⁵` ⇒ ưu tiên DFS bằng stack trong Python.
- Viết sẵn hàm `run_commands` để test nhanh mọi bài "Design".

## 15. Biến thể của bài

1. **`getSuccessor(name)`**: trả người còn sống đứng ngay sau `name` trong thứ tự (bài tập 5).
2. **Thoái vị (abdicate)**: một người tự loại mình và cả nhánh con cháu — khi đó *xóa cứng* là đúng.
3. **Kế vị theo trưởng nữ / chỉ nam giới** — thêm thuộc tính, lọc khi duyệt.
4. **LC 589 — N-ary Tree Preorder Traversal**: phần lõi của bài này.
5. **LC 1166 — Design File System**, **LC 588 — Design In-Memory File System**: thiết kế cây bằng `dict`.
6. **Truy vấn thứ hạng kế vị** của một người (vị trí trong danh sách) nhanh — cần Euler tour + cấu trúc đếm (Fenwick) nếu có nhiều truy vấn.

## 16. So sánh các cách

| Cách | Ý tưởng | getInheritanceOrder | Ghi chú | Khi nào dùng |
|---|---|---|---|---|
| 1. Successor nguyên văn | mô phỏng định nghĩa + set curOrder | O(N²) worst | đúng nghĩa đen | kiểm chứng hiểu đề |
| 2. DFS đệ quy | preorder, bỏ tên người chết | O(N) | có thể RecursionError | code ngắn, cây nông |
| 3. DFS stack | preorder bằng stack, `reversed` | O(N) | an toàn mọi độ sâu | **nộp bài** |
| 4. Con trỏ cha | đi xuống/lên với chỉ số con kế tiếp | O(N) | không cần stack | khi cần "Successor" từng bước |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** Kết quả `getInheritanceOrder` sau chuỗi: `ThroneInheritance("k")`, `birth("k","a")`, `birth("a","b")`, `birth("k","c")`, `birth("b","d")`, `death("a")`, `death("d")`?

**Bài 2 (tính tay):** Với cây ở Bài 1 (chưa ai chết), viết các lần gọi `Successor` và giá trị trả về của từng lần.

**Bài 3 (điền TODO):** Thêm vào lớp hàm `generation(name)` trả về số đời từ vua tới `name` (vua là 0).

**Bài 4 (tự viết lại):** Viết hàm độc lập `preorder_alive(children, king, dead)` theo Cách 4 (con trỏ cha), nhận vào `children` và tự xây `parent`.

**Bài 5 (mini challenge):** Viết lớp `ThroneWithSuccessor` có thêm `getSuccessor(name)` trả người **còn sống** đứng ngay sau `name` trong thứ tự kế vị (hoặc `None`), không xây cả danh sách.

In [22]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
class ThroneWithGeneration(ThroneParentWalk):
    def generation(self, name):
        depth = 0
        # TODO: leo theo self.parent cho tới khi gặp vua (parent là None), đếm số bước
        return depth

In [23]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def preorder_alive(children, king, dead):
    # TODO
    pass

class ThroneWithSuccessorTodo(ThroneParentWalk):
    def getSuccessor(self, name):
        # TODO
        pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: vẽ cây. Mức 2: preorder `k, a, b, d, c`. Mức 3: bỏ `a`, `d` ⇒ `[k, b, c]`.

**Bài 2.** Mức 1: bắt đầu `curOrder = [k]`. Mức 2: `Successor(k)=a`, `Successor(a)=b`, `Successor(b)=d`. Mức 3: `Successor(d)`: d không con ⇒ lên b (hết) ⇒ lên a (hết) ⇒ lên k ⇒ `c`; `Successor(c)` ⇒ lên k (hết, là vua) ⇒ `null`.

**Bài 3.** Mức 1: dùng `self.parent`. Mức 2: `while self.parent[name] is not None`. Mức 3: mỗi vòng `name = self.parent[name]; depth += 1`.

**Bài 4.** Mức 1: `parent[c] = p` cho mọi `p, c`. Mức 2: dùng `nxt` như Cách 4. Mức 3: nhớ `parent[king] = None` để vòng lặp dừng.

**Bài 5.** Mức 1: người kế tiếp trong preorder: con đầu tiên nếu có; nếu không, leo lên tìm tổ tiên có "em kế tiếp". Mức 2: để biết "em kế tiếp" cần vị trí của mỗi người trong danh sách con của cha (`idx[c]`). Mức 3: lặp lại thao tác "người kế tiếp" cho tới khi gặp người còn sống hoặc hết cây.

## 19. Đáp án tham khảo cho bài tập

In [24]:
# Bài 1
ops = ["ThroneInheritance", "birth", "birth", "birth", "birth", "death", "death", "getInheritanceOrder"]
args = [["k"], ["k", "a"], ["a", "b"], ["k", "c"], ["b", "d"], ["a"], ["d"], []]
b1 = run_commands(ThroneInheritance, ops, args)[-1]
print("Bài 1:", b1)
assert b1 == ["k", "b", "c"]

# Bài 2: in các bước Successor bằng cách 1 (không ai chết)
tb = ThroneBrute("k")
for p, c in [("k", "a"), ("a", "b"), ("k", "c"), ("b", "d")]:
    tb.birth(p, c)
order, seen = ["k"], {"k"}
while True:
    nx = tb._successor(order[-1], seen)
    print(f"Successor({order[-1]}) = {nx}")
    if nx is None: break
    order.append(nx); seen.add(nx)
assert order == ["k", "a", "b", "d", "c"]

Bài 1: ['k', 'b', 'c']
Successor(k) = a
Successor(a) = b
Successor(b) = d
Successor(d) = c
Successor(c) = None


In [25]:
# Bài 3
class ThroneWithGenerationSol(ThroneParentWalk):
    def generation(self, name):
        depth = 0
        while self.parent[name] is not None:
            name = self.parent[name]; depth += 1
        return depth

tg = ThroneWithGenerationSol("k")
for p, c in [("k", "a"), ("a", "b"), ("k", "c"), ("b", "d")]:
    tg.birth(p, c)
assert [tg.generation(x) for x in "kabcd"] == [0, 1, 2, 1, 3]
print("Bài 3: OK")

Bài 3: OK


In [26]:
# Bài 4
def preorder_alive_sol(children, king, dead):
    parent = {king: None}
    for p, kids in children.items():
        for c in kids:
            parent[c] = p
    nxt = defaultdict(int)
    order = [] if king in dead else [king]
    x = king
    while x is not None:
        kids = children.get(x, [])
        if nxt[x] < len(kids):
            c = kids[nxt[x]]; nxt[x] += 1
            if c not in dead: order.append(c)
            x = c
        else:
            x = parent[x]
    return order

rnd = random.Random(4)
for _ in range(200):
    ops, args = random_script(rnd, rnd.randint(1, 40))
    t = ThroneStack(args[0][0])
    for op, a in zip(ops[1:], args[1:]):
        if op != "getInheritanceOrder":
            getattr(t, op)(*a)
    assert preorder_alive_sol(dict(t.children), t.king, t.dead) == t.getInheritanceOrder()
print("Bài 4: OK")

Bài 4: OK


In [27]:
# Bài 5
class ThroneWithSuccessor(ThroneParentWalk):
    def __init__(self, kingName):
        super().__init__(kingName)
        self.idx = {kingName: 0}                 # vị trí trong danh sách con của cha

    def birth(self, parentName, childName):
        self.idx[childName] = len(self.children[parentName])
        super().birth(parentName, childName)

    def _next_in_preorder(self, x):
        if self.children[x]:
            return self.children[x][0]
        while x != self.king:
            p = self.parent[x]
            j = self.idx[x] + 1
            if j < len(self.children[p]):
                return self.children[p][j]
            x = p
        return None

    def getSuccessor(self, name):
        x = self._next_in_preorder(name)
        while x is not None and x in self.dead:
            x = self._next_in_preorder(x)
        return x

rnd = random.Random(5)
for _ in range(200):
    ops, args = random_script(rnd, rnd.randint(1, 40))
    t = ThroneWithSuccessor(args[0][0])
    for op, a in zip(ops[1:], args[1:]):
        if op != "getInheritanceOrder":
            getattr(t, op)(*a)
    # đối chiếu: người còn sống kế tiếp trong preorder đầy đủ (kể cả người chết)
    full = preorder_alive_sol(dict(t.children), t.king, set())
    for i, x in enumerate(full):
        expect = next((y for y in full[i + 1:] if y not in t.dead), None)
        assert t.getSuccessor(x) == expect
print("Bài 5: OK")

Bài 5: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template thiết kế cây kế vị:**

```python
self.children = defaultdict(list)   # append theo thứ tự sinh
self.dead = set()                   # xóa mềm

def getInheritanceOrder(self):
    order, stack = [], [self.king]
    while stack:
        x = stack.pop()
        if x not in self.dead: order.append(x)
        stack.extend(reversed(self.children[x]))   # luôn đi vào con
    return order
```

**Checklist:**
- [ ] `death` chỉ đánh dấu, không xóa khỏi cây?
- [ ] DFS vẫn đi vào con của người đã chết?
- [ ] Con giữ đúng thứ tự sinh (không sort, không dùng `set`)?
- [ ] Đẩy con vào stack bằng `reversed`?
- [ ] Không dùng đệ quy sâu (chuỗi `10⁵` đời)?
- [ ] Đã test: vua chết, chỉ có vua, sinh con sau khi có người chết, chuỗi rất sâu?